# Receding-horizon LQR

This notebook reproduces the discrete-time example from Lecture 5. It builds and solves each finite-horizon problem with CVXPY and OSQP, applies only the first optimal input, and repeats at the next sample. All code needed for the experiments is in this notebook.

We will compare horizons $N=2,\ldots,10$ with $P=I$, then compare a short horizon with an algebraic Riccati equation (ARE) terminal cost against a longer horizon with $P=I$.


## 1. Setup

Run the next cell in Google Colab to install the required packages. It skips installation in a local Python session, where the packages can be installed from this directory's `requirements.txt` if needed. Run the remaining cells in order.


In [ ]:
# Optional package installation:
%pip install 'numpy>=1.26' 'scipy>=1.11' 'matplotlib>=3.7' 'cvxpy>=1.6' 'osqp>=1.0'

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import solve_discrete_are

np.set_printoptions(precision=6, suppress=True)

## 2. Plant and finite-horizon objective

Slide 46 uses the sampled plant

$$
x_{k+1}=
\begin{bmatrix}1.988&-0.998\\1&0\end{bmatrix}x_k+
\begin{bmatrix}0.125\\0\end{bmatrix}u_k.
$$

We start from $x_0=[0.4,\;0.4]^\mathsf{T}$, plot the first state as the output, and use $Q=I$, $R=1$. For horizon $N$, the batch optimizer minimizes

$$
J_N=\sum_{j=0}^{N-1}\left(x_j^\mathsf{T}Qx_j+u_j^\mathsf{T}Ru_j\right)
+x_N^\mathsf{T}Px_N
$$

subject to the dynamics. Here $N$ counts **control moves**, following the lecture's written equation.


In [ ]:
# Model, cost weights, and initial state from the lecture example.
A = np.array([[1.988, -0.998], [1.0, 0.0]])
B = np.array([[0.125], [0.0]])
Q = np.eye(2)
R = np.eye(1)
P_IDENTITY = np.eye(2)
X_INITIAL = np.array([0.4, 0.4])

HORIZON_SIMULATION_STEPS = 150
TERMINAL_SIMULATION_STEPS = 50


## 3. Build the batch optimization problem

The decision variables contain all $N+1$ predicted states and all $N$ predicted inputs. A CVXPY parameter holds the latest measured state, so the same problem can be solved repeatedly without rebuilding it. The equality constraints encode the plant dynamics; the quadratic objective encodes the stage and terminal costs.


In [ ]:
def make_batch_problem(horizon: int, terminal_weight: np.ndarray):
    """Build the N-move quadratic program for one terminal weight."""
    if horizon < 1:
        raise ValueError("horizon must contain at least one control move")

    x_measured = cp.Parameter(2)
    x = cp.Variable((2, horizon + 1))  # x_0, ..., x_N
    u = cp.Variable((1, horizon))      # u_0, ..., u_{N-1}

    constraints = [x[:, 0] == x_measured]
    cost = 0
    for j in range(horizon):
        constraints.append(x[:, j + 1] == A @ x[:, j] + B @ u[:, j])
        cost += cp.quad_form(x[:, j], Q) + cp.quad_form(u[:, j], R)
    cost += cp.quad_form(x[:, horizon], terminal_weight)

    problem = cp.Problem(cp.Minimize(cost), constraints)
    return problem, x_measured, u


## 4. Simulate receding-horizon feedback

At every sample we set the measured state, solve the full batch problem, apply $u_0^\star$, and advance the discrete-time plant. Re-solving after every step makes the controller a feedback law, even though each individual optimization predicts an open-loop input sequence.


In [ ]:
def simulate_receding_horizon(
    horizon: int, terminal_weight: np.ndarray, simulation_steps: int
):
    """Return closed-loop states and applied inputs over the requested steps."""
    if simulation_steps < 1:
        raise ValueError("simulation_steps must be positive")

    problem, x_measured, u = make_batch_problem(horizon, terminal_weight)
    states = np.empty((simulation_steps + 1, 2))
    inputs = np.empty(simulation_steps)
    states[0] = X_INITIAL

    for k in range(simulation_steps):
        x_measured.value = states[k]
        problem.solve(
            solver=cp.OSQP, warm_start=True, eps_abs=1e-8, eps_rel=1e-8
        )
        if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(
                f"OSQP failed at step {k}, horizon {horizon}: {problem.status}"
            )

        inputs[k] = u.value[0, 0]  # Apply only the first planned move.
        states[k + 1] = A @ states[k] + B[:, 0] * inputs[k]

    return states, inputs


## 5. Experiment 1: effect of horizon length

With $P=I$, simulate $N=2,\ldots,10$ for 150 samples. The plot follows the form of slide 46: output (the first state) versus sample index. At 10 Hz, 10 samples correspond to one second.


In [ ]:
horizon_responses = {
    horizon: simulate_receding_horizon(
        horizon, P_IDENTITY, HORIZON_SIMULATION_STEPS
    )
    for horizon in range(2, 11)
}

fig, ax = plt.subplots(figsize=(9, 5))
for horizon, (states, _) in horizon_responses.items():
    ax.plot(np.arange(len(states)), states[:, 0],
            label=f"N = {horizon}", linewidth=1.7)
ax.set(
    title="Closed-loop response: impact of horizon length ($P=Q=I$, $R=1$)",
    xlabel="Time (samples)", ylabel="Output $x_1$",
    xlim=(0, HORIZON_SIMULATION_STEPS), ylim=(-0.4, 0.4),
)
ax.grid(True, alpha=0.35)
ax.legend(ncol=3)
fig.tight_layout()
plt.show()


**Reading the plot.** Short horizons account for less of the plant's future motion, so their closed-loop responses differ more from the longer-horizon responses.


## 6. Choose the terminal cost from the ARE

Lecture 5 presents a method for choosing the terminal cost matrix as the infinite-horizon LQR cost matrix $P_\infty$, which solves

$$
P_\infty=A^\mathsf{T}P_\infty A+Q
-A^\mathsf{T}P_\infty B
(R+B^\mathsf{T}P_\infty B)^{-1}
B^\mathsf{T}P_\infty A.
$$

SciPy computes this matrix with `solve_discrete_are`. The residual below checks it against the displayed equation.


In [ ]:
P_ARE = solve_discrete_are(A, B, Q, R)
are_rhs = (
    A.T @ P_ARE @ A + Q
    - A.T @ P_ARE @ B
    @ np.linalg.solve(R + B.T @ P_ARE @ B, B.T @ P_ARE @ A)
)
print("ARE terminal matrix P =\n", P_ARE)
print("ARE residual (Frobenius norm):", np.linalg.norm(P_ARE - are_rhs))


## 7. Experiment 2: terminal cost versus horizon length

Run each case for 50 samples:

1. $N=5,\ P=I$: the short-horizon baseline.
2. $N=5,\ P=P_\infty$: the same short horizon with the ARE terminal cost.
3. $N=10,\ P=I$: a longer horizon with the original terminal cost.

Comparing the second and third cases shows how a suitable terminal cost can compensate for a shorter prediction horizon.


In [ ]:
terminal_responses = {
    "$N=5$, $P=I$": simulate_receding_horizon(
        5, P_IDENTITY, TERMINAL_SIMULATION_STEPS
    ),
    "$N=5$, $P=P_{\\infty}$ (ARE)": simulate_receding_horizon(
        5, P_ARE, TERMINAL_SIMULATION_STEPS
    ),
    "$N=10$, $P=I$": simulate_receding_horizon(
        10, P_IDENTITY, TERMINAL_SIMULATION_STEPS
    ),
}

fig, (ax_y, ax_u) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for label, (states, inputs) in terminal_responses.items():
    ax_y.plot(np.arange(len(states)), states[:, 0],
              label=label, linewidth=1.8)
    ax_u.step(np.arange(len(inputs)), inputs, where="post",
              label=label, linewidth=1.4)

ax_y.set(ylabel="Output $x_1$",
         title="Terminal cost and horizon comparison (50 steps)")
ax_u.set(xlabel="Time (samples)", ylabel="Input $u$",
         xlim=(0, TERMINAL_SIMULATION_STEPS))
for ax in (ax_y, ax_u):
    ax.grid(True, alpha=0.35)
    ax.legend()
fig.tight_layout()
plt.show()


For a numerical comparison, compute the **realized** 50-step stage cost $\sum_{k=0}^{49}(x_k^\mathsf{T}Qx_k+u_k^\mathsf{T}Ru_k)$. The same metric is used for all three closed-loop runs; no terminal penalty is added to this reported value.


In [ ]:
for label, (states, inputs) in terminal_responses.items():
    stage_cost = (
        np.einsum("ki,ij,kj->", states[:-1], Q, states[:-1])
        + R[0, 0] * np.sum(inputs**2)
    )
    print(f"{label}: realized 50-step stage cost = {stage_cost:.6f}")


**Interpretation.** For this unconstrained LQR problem, $P_\infty$ represents the optimal cost beyond the prediction horizon. Choosing it as the terminal weight gives the infinite-horizon feedback law even at $N=5$. The $N=10,\ P=I$ response approaches that behavior by planning farther ahead.
